# Day 13 · 训练环境与显存工程

**配套讲义**: [`days/day-13.md`](../days/day-13.md) ｜ **需要 GPU（云机器）**

把「显存四分账」——权重 / 梯度 / 优化器状态 / 激活值 —— 变成一张能直接指导租卡的表格：3B·7B × 全参·LoRA·QLoRA × seq 2048·4096。

> 📌 本 notebook 由 `scripts/gen_days.py` 生成 —— **别手改**，
> 要改内容请改 `scripts/daygen/w3.py` 后重跑脚本。

## 0. 环境检查

In [ ]:
import sys, torch
print("python :", sys.version.split()[0])
print("torch  :", torch.__version__)
print("cuda   :", torch.version.cuda, "| available:", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"gpu    : {p.name}  {p.total_memory / 1024**3:.0f} GB")
    print("bf16   :", torch.cuda.is_bf16_supported())
else:
    print("⚠️  没有 GPU —— 这一天的训练/推理跑不了。先看 docs/13-hardware-and-cost.md 租机器")

## 1. 跑全矩阵

In [ ]:
import subprocess, sys
out = subprocess.run([sys.executable, "../scripts/estimate_vram.py", "--all"],
                     capture_output=True, text=True)
print(out.stdout or out.stderr)

## 2. 手算一遍，验证脚本没骗你

显存四分账里，**静态三项**（权重/梯度/优化器）是可以手算的。
激活值必须靠经验区间 —— 这就是为什么脚本要同时打印两个数。

In [ ]:
P = 3.0e9                      # 3B 总参数
GB = 1024 ** 3
trainable_ratio = 0.005        # LoRA r=16 大约 0.5% 可训练参数

weights = P * 0.5 / GB                      # 4-bit 权重
lora    = P * trainable_ratio * 2 / GB      # A/B 矩阵，bf16
grad    = lora                              # 只有 LoRA 参数有梯度
opt     = P * trainable_ratio * 8 / GB      # AdamW 两个状态 × 4 字节

print(f"权重(4bit) {weights:6.2f} GB")
print(f"LoRA A/B   {lora:6.3f} GB")
print(f"梯度       {grad:6.3f} GB")
print(f"优化器     {opt:6.3f} GB")
print(f"静态合计   {weights+lora+grad+opt:6.2f} GB   ← 剩下的全给激活值 + logits + KV")

## 3. 自己改参数做敏感性实验

把 `seq_len` 从 2048 拉到 4096、把图片从 1 张改成 2 张，看哪一项涨得最凶。
**结论应该指向：图片 token 数比 batch 更影响显存** —— 这是 VLM 训练和纯文本训练最大的不同。

In [ ]:
import subprocess, sys
for args in (["--model", "3b", "--method", "qlora", "--seq", "2048"],
             ["--model", "3b", "--method", "qlora", "--seq", "4096"],
             ["--model", "7b", "--method", "qlora", "--seq", "2048"]):
    r = subprocess.run([sys.executable, "../scripts/estimate_vram.py", *args],
                       capture_output=True, text=True)
    tail = [l for l in r.stdout.splitlines() if "解析估算合计" in l or "实测经验区间" in l]
    print(" ".join(args[-2:]), "→", " | ".join(t.strip() for t in tail))

## 验收清单

- [ ] `--all` 矩阵能说清每一格为什么差这么多（尤其「3B 全参 63 GB vs QLoRA 5 GB」）
- [ ] `--suggest` 能给出卡型建议，并且知道为什么推荐 3090/4090 而不是 T4
- [ ] 能不看资料推导出「3B QLoRA 在 16 GB 卡上必须开 gradient checkpointing」
- [ ] **租卡决定已经做了**（写进今日打卡：准备租哪张卡、为什么、预算多少）

**卡住了？** 回看 [`days/day-13.md`](../days/day-13.md) 第五节「容易踩的坑」。

> **明天**：`days/day-14.md` —— LoRA 原理，手写低秩 A/B 矩阵